# Semantic Chunking: RAG

## Why Semantic Chunking?

In a standard RAG pipeline (see `CH-2_RAG/1_RAG.ipynb`), documents are split into **fixed-size chunks** using a character or token count (e.g. `RecursiveCharacterTextSplitter`). This is simple but has a key weakness: **chunk boundaries are arbitrary** — a chunk might cut a paragraph in half, mixing two unrelated ideas or splitting one idea across two chunks.

**Semantic Chunking** solves this by splitting text based on **meaning** rather than length:

| | Fixed-Size Chunking | Semantic Chunking |
|---|---|---|
| **Split by** | Character / token count | Embedding similarity between sentences |
| **Chunk size** | Uniform | Variable (depends on content) |
| **Boundary quality** | May break mid-idea | Breaks between distinct ideas |
| **Retrieval quality** | Good for general use | Better for nuanced questions |
| **Cost** | Cheaper (no embeddings at split time) | Requires embedding each sentence to decide splits |

## How Semantic Chunking Works

1. **Sentence-level embeddings** — Each sentence in the document is embedded individually.
2. **Pairwise similarity** — The cosine similarity between consecutive sentence embeddings is computed.
3. **Breakpoint detection** — Where the similarity drops below a threshold, a chunk boundary is inserted.
4. **Chunk formation** — Consecutive sentences that stay above the threshold are grouped into a single chunk.

The result: each chunk is a **semantically coherent unit** — all sentences in a chunk are about the same topic.

In [2]:
#Prerequisites
# uv add langchain_openai
# uv add langchain
# uv add langchain_community
# uv add pypdf

import os
from langchain_openai import ChatOpenAI #
from langchain_community.document_loaders import PyPDFLoader # Loads the PDF file
from langchain_experimental.text_splitter import SemanticChunker
 # Splits the text into chunks
from dotenv import load_dotenv # Loads the environment variables


load_dotenv()

True

## **Step 1: Load Document & Create Semantic Chunks**

**Key imports:**
- `PyPDFLoader` — reads the PDF and returns one `Document` per page
- `SemanticChunker` (from `langchain_experimental`) — splits text by meaning, not by character count

We first join all pages into a single string (`full_text`) because semantic chunking needs the entire document to compute sentence-level similarities.

In [3]:
docs = PyPDFLoader("../CH-2_RAG/NovaS.pdf").load()

full_text = "\n".join([doc.page_content for doc in docs])


In [5]:
from langchain_openai import OpenAIEmbeddings

embed_model = OpenAIEmbeddings(model = 'text-embedding-3-small')
chunker = SemanticChunker(
    embed_model,
    breakpoint_threshold_type = 'percentile',
    breakpoint_threshold_amount = 60
)

semantic_chunks = chunker.create_documents([full_text])
semantic_chunks

[Document(metadata={}, page_content='NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future.'),
 Document(metadata={}, page_content='At the \nbeginning, the company did not have large investments or a big office.'),
 Document(metadata={}, page_content='Instead, it started \nwith only six employees working together in a small shared workspace. The founders were \nnot focused on becoming successful overnight.'),
 Document(metadata={}, page_content='Their main goal was to build strong \ntechnical knowledge, gain practical experience, and slowly grow by delivering real value to \ntheir clients. Most of the early work involved helping small companies understand their \nexisting data and use simple reporting 

### SemanticChunker Parameters

| Parameter | Value Used | Meaning |
|---|---|---|
| `embed_model` | `text-embedding-3-small` | The embedding model used to embed each sentence for similarity comparison |
| `breakpoint_threshold_type` | `'percentile'` | How the similarity drop threshold is calculated. `'percentile'` means: "split when the similarity drop is in the top X percentile of all drops" |
| `breakpoint_threshold_amount` | `60` | 60th percentile — any similarity drop larger than 60% of all observed drops triggers a new chunk |

**Other threshold types available:**
- `'standard_deviation'` — split when drop exceeds X standard deviations from the mean
- `'interquartile'` — split based on IQR outlier detection
- `'gradient'` — split based on the rate of change in similarity

> **Lower threshold = more chunks (finer splits)**. A threshold of 60 is moderate — it produces chunks that are topically focused without being too granular.

## **Step 2 & 3: Create Embeddings and Store in Vector DB**

Once we have semantically coherent chunks, the rest of the RAG pipeline is identical to fixed-size chunking:

1. **Embed each chunk** using the same embedding model (`text-embedding-3-small`)
2. **Store in ChromaDB** — a lightweight vector database that persists to `./vector_db_semantic`

`Chroma.from_documents()` handles both steps in one call — it embeds each `Document` and inserts the vectors into the DB.

In [7]:
from langchain_community.vectorstores import Chroma

Chroma.from_documents(
    documents = semantic_chunks,
    embedding = embed_model,
    persist_directory = './vector_db_semantic'
)


## **Step 4: Connection to Vector DB and Retrieval**

To query the vector DB, we open a connection to the persisted Chroma store using the same embedding function. Then `similarity_search(query, k=N)` does the following under the hood:

1. **Embeds the query** using `text-embedding-3-small`
2. **Computes cosine similarity** between the query vector and every stored chunk vector
3. **Returns the top-k** most similar chunks as `Document` objects

**Why `k` matters:** Too few results may miss relevant context; too many may introduce noise. `k=3` is a good starting point for focused questions.

In [8]:
chroma_db_conn = Chroma(
    embedding_function = embed_model,
    persist_directory = './vector_db_semantic'
)

/var/folders/25/h441spjj4nl4qg_jjtw1yxhh0000gn/T/ipykernel_39500/2923992167.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_db_conn = Chroma(


In [9]:
chroma_db_conn.similarity_search('When NovaSphere Technologies was founded?', k=3)

[Document(metadata={}, page_content='NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future.'),
 Document(metadata={}, page_content='The founders also started planning \nthe next stage of growth, which included expanding into new regions and working with \ninternational clients. Today, NovaSphere Technologies is considered a reliable organization that provides data \nengineering and analytics services to companies from different industries such as finance, \nhealthcare, retail, and e-commerce. Even though the company has grown significantly \nsince 2016, the original vision has not changed. The focus is still on learning continuously, \nimproving the quality of work, and helping organizations make bette

In [11]:
chroma_db_conn.similarity_search('By 2019, how many employees did NovaSphere Technologies have?', k=3)

[Document(metadata={'creationdate': '2026-03-31T11:24:15-03:00', 'producer': 'Microsoft® Word for Microsoft 365', 'page_label': '3', 'author': 'Ansh Lamba', 'source': 'NovaS.pdf', 'moddate': '2026-03-31T11:24:15-03:00', 'total_pages': 3, 'creator': 'Microsoft® Word for Microsoft 365', 'page': 2}, page_content='By 2024, NovaSphere Technologies had become a well-known name among mid-sized'),
 Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-31T11:24:15-03:00', 'page': 2, 'author': 'Ansh Lamba', 'moddate': '2026-03-31T11:24:15-03:00', 'page_label': '3', 'creator': 'Microsoft® Word for Microsoft 365', 'source': 'NovaS.pdf', 'total_pages': 3}, page_content='By 2024, NovaSphere Technologies had become a well-known name among mid-sized'),
 Document(metadata={'creationdate': '2026-03-31T11:24:15-03:00', 'page': 0, 'source': 'NovaS.pdf', 'moddate': '2026-03-31T11:24:15-03:00', 'author': 'Ansh Lamba', 'page_label': '1', 'producer': 'Microsoft® Word for

## **Step 5: Pass Context to LLM and Receive Responses**

This is the **generation** step — where RAG becomes "Retrieval-**Augmented** Generation":

1. Accept a user question via `input()`
2. Retrieve the top-10 most relevant semantic chunks from ChromaDB
3. Concatenate the chunk contents into a single context string
4. Pass both the question and context to the LLM in a single prompt

The LLM does **not** need to know the answer from its training data — it synthesizes an answer purely from the retrieved context. This grounds the response in the source document and reduces hallucination.

> **Note:** `temperature=0` makes the LLM deterministic — it always picks the highest-probability token, which is ideal for factual Q&A.

In [10]:
# temperature is parameter to control randomness of the model, 0 is deterministic and 1 is random. Anything in between means the model will be somewhere in between.
llm = ChatOpenAI(model = 'gpt-3.5-turbo', temperature = 0) 
llm

ChatOpenAI(profile={'name': 'GPT-3.5-turbo', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'attachment': False, 'temperature': True, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x166545e80>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x166545370>, root_client=<openai.OpenAI object at 0x1662de6c0>, root_async_client=<openai.AsyncOpenAI object at 0x166547d70>, temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usa

In [11]:
user_query = input("Enter your question:")
relevant_chunks = chroma_db_conn.similarity_search(user_query, k=10)

relevant_chunks_content = []
for i, chunk in enumerate(relevant_chunks):
    relevant_chunks_content.append(chunk.page_content)
relevant_chunks_content = str(relevant_chunks_content)

print(relevant_chunks_content)
llm.invoke(f"{user_query}, Use the following context to answer the question: {relevant_chunks_content}")

['NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future.', 'The founders also started planning \nthe next stage of growth, which included expanding into new regions and working with \ninternational clients. Today, NovaSphere Technologies is considered a reliable organization that provides data \nengineering and analytics services to companies from different industries such as finance, \nhealthcare, retail, and e-commerce. Even though the company has grown significantly \nsince 2016, the original vision has not changed. The focus is still on learning continuously, \nimproving the quality of work, and helping organizations make better decisions using data. The company believes that the future of business

AIMessage(content='Novasphere was founded in 2016.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 964, 'total_tokens': 974, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DUWgQnfKVDl1shQ3gFm074zEhKoVp', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d8bda-7d4f-79d3-9b51-be210ce6d6cf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 964, 'output_tokens': 10, 'total_tokens': 974, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})